# Módulo AI2 · Aplicaciones con LLMs — Práctica guiada

**Producto:** el *asistente de pedidos de TiendaSur*.

1. System prompt con reglas accionables (dominio + escape)
2. Extraer `Pedido` de un email de cliente (Pydantic)
3. Ferramenta `get_pedido(id)` y bucle de herramienta
4. Streaming de la respuesta final

> **Clave de API.** Necesitas `ANTHROPIC_API_KEY`. En Colab: *Secretos* (icono llave) con ese nombre.
> En local: `export ANTHROPIC_API_KEY=sk-...`. **Nunca la escribas en el notebook.**
> Sin clave, las celdas marcadas **[API]** se saltan limpio (el notebook se ejecuta entero igualmente).


In [ ]:
import os
import json
import textwrap
from typing import Optional, Literal

# Colab: userdata
try:
    import google.colab  # noqa
    from google.colab import userdata
    os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY") or "")
except ImportError:
    pass

TIENE_CLAVE = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("clave de API presente:", TIENE_CLAVE)
if not TIENE_CLAVE:
    print("Sin clave: las celdas [API] se saltarán y seguirá el flujo con datos simulados.")

In [ ]:
from pydantic import BaseModel, Field

class Pedido(BaseModel):
    """Un pedido extraído de un email de cliente."""
    producto: str = Field(description="nombre del producto pedido")
    unidades: int = Field(ge=1, description="número de unidades, >=1")
    urgente: bool = Field(description="marcar solo si el cliente lo pide explícitamente")
    email_resumen: str = Field(description="resumen de una línea del email")

print(Pedido.model_json_schema()["properties"].keys())

## 2.2 · System prompt con reglas accionables

In [ ]:
SYSTEM = textwrap.dedent("""
Eres el asistente de pedidos de TiendaSur.

Dominio:
- Responde SOLO sobre pedidos, envíos y facturas de TiendaSur.
- Si el usuario pregunta otra cosa, responde exactamente: "Te paso a soporte".

Reglas:
- NUNCA inventes datos de un pedido: usa la herramienta `get_pedido`.
- Si el usuario pide cambiar algo (dirección, fecha), di: "El cambio debe hacerlo en la app".
- Tonos: cordial de 2-4 frases, sin emojis, sin markdown de encabezados.
""")
print(SYSTEM[:280], "…")

**Lectura:** 3 secciones — dominio con **frase exacta de escape**, reglas accionables (nada de "sé útil") y tono. Todo es comprobable: si el modelo falla, sabes cuál regla falló.


## [API] Cliente y utilidades

In [ ]:
import anthropic

MODELO = "claude-sonnet-4-5"

def cliente():
    return anthropic.Anthropic()   # lee ANTHROPIC_API_KEY

def preguntar(mensaje: str, system: str = SYSTEM, max_tokens: int = 500) -> str:
    """Llamada simple: devuelve el texto de la respuesta."""
    if not TIENE_CLAVE:
        return "[SALTADA: sin clave de API. Con clave, aquí verías la respuesta real.]"
    c = cliente()
    msg = c.messages.create(
        model=MODELO, system=system, max_tokens=max_tokens,
        messages=[{"role": "user", "content": mensaje}],
    )
    return msg.content[0].text

## 2.3 · Extracción estructurada de un email

Pedimos JSON con el schema del Pydantic **en el prompt** + validamos con `model_validate_json`. Si falla, re-llamamos con el error.


In [ ]:
EMAIL = """Hola, el viernes pedi 3 tazas de cafe TiendaSur y necesito que llegue antes del jueves.
Mi numero de pedido es 1042 y tengo prisa. Gracias. Mario"""

SCHEMA = json.dumps(Pedido.model_json_schema(), indent=1)

from typing import Optional

def extraer_pedido(email: str) -> Optional[Pedido]:
    """[API] Extrae una Pedido del email; re-llama si el JSON no valida."""
    if not TIENE_CLAVE:
        print("[SALTADA: sin clave]")
        return None
    prompt = f"""Extrae los datos del email del jugador. Devuelve SOLO JSON válido con este schema:

{SCHEMA}

Email:
{email}"""
    for intento in range(2):
        msg = cliente().messages.create(model=MODELO, max_tokens=300,
                                        messages=[{"role": "user", "content": prompt}])
        try:
            return Pedido.model_validate_json(msg.content[0].text)
        except Exception as e:
            prompt = f"""Tu respuesta anterior no validó: {e}
Vuelve a responder con SOLO JSON válido. Schema: {SCHEMA}. Email: {email}"""
    return None

pedido = extraer_pedido(EMAIL)
if pedido:
    print("producto :", pedido.producto)
    print("unidades :", pedido.unidades)
    print("urgente  :", pedido.urgente)
    print("resumen  :", pedido.email_resumen)

**Lectura:** sin el schema, un modelo puede responder `"producto": "café", "unidades": "tres"` (string). Con schema + Pydantic: error nombrado, re-llamo y llega `3`. Es la diferencia entre "aparentemente funciona" y "funciona en producción".


## 2.4 · Ferramenta: el modelo pide, tú ejecutas

In [ ]:
# Datos simulados de TiendaSur (en la vida real: tu BD)
PEDIDOS = {
    1041: {"producto": "Café grano 1kg", "estado": "entregado", "fecha": "2024-05-10"},
    1042: {"producto": "Tazas TiendaSur", "estado": "preparando",  "fecha": "2026-02-06"},
    1043: {"producto": "Prensa francesa", "estado": "enviado",  "fecha": "2026-02-04"},
}

def get_pedido(pedido_id: int) -> dict:
    return PEDIDOS.get(pedido_id, {"error": f"pedido {pedido_id} no existe"})

TOOL_SCHEMA = [{
    "name": "get_pedido",
    "description": "Devuelve el estado y detalle de un pedido de TiendaSur por su id.",
    "input_schema": {
        "type": "object",
        "properties": {"pedido_id": {"type": "integer"}},
        "required": ["pedido_id"],
    },
}]
print("herramienta declarada:", TOOL_SCHEMA[0]["name"])

In [ ]:
def conversacion_con_tools(mensaje: str):
    """[API] Bucle: modelo pide herramienta -> tú ejecutas -> modelo responde."""
    if not TIENE_CLAVE:
        print("[SALTADA: sin clave]")
        print("(con API: el modelo pediría get_pedido(1042) y respondería con el estado real)")
        return
    c, mensajes = cliente(), [{"role": "user", "content": mensaje}]
    for vuelta in range(3):           # límite de bucles de herramienta
        resp = c.messages.create(model=MODELO, system=SYSTEM, max_tokens=600,
                                 tools=TOOL_SCHEMA, messages=mensajes)
        if resp.stop_reason != "tool_use":
            print(resp.content[0].text)
            return
        bloque_tool = next(b for b in resp.content if b.type == "tool_use")
        print(f"→ el modelo pide {bloque_tool.name}({bloque_tool.input})")
        resultado = get_pedido(**bloque_tool.input)
        mensajes.append({"role": "assistant", "content": resp.content})
        mensajes.append({"role": "user",
                         "content": [{"type": "tool_result", "tool_use_id": bloque_tool.id,
                                      "content": json.dumps(resultado)}]})
    print("(límite de vueltas alcanzo: revisa la herramienta)")

conversacion_con_tools("¿En qué está mi pedido 1042?")

**Lectura:** el **modelo decide** llamar a la herramienta; **tu función** corre y devuelve el JSON; el modelo **continúa** con ese dato. Es un bucle que puede iterar (p.ej. `get_pedido` → `get_envio`).

Guardas el límite de vueltas (3) para no disparar llamadas infinitas en producción.


## 2.5 · Streaming

In [ ]:
def stream_respuesta(mensaje: str):
    """[API] Streaming: imprime los tokens a medida que llegan."""
    if not TIENE_CLAVE:
        print("[SALTADA: sin clave] — con clave, aquí verías la respuesta llegando palabra a palabra.")
        return
    c = cliente()
    with c.messages.stream(model=MODELO, system=SYSTEM, max_tokens=500,
                           messages=[{"role": "user", "content": mensaje}]) as s:
        for texto in s.text_stream:
            print(texto, end="", flush=True)
    print()

stream_respuesta("¿Puedo cambiar la dirección del pedido 1042?")

**Lectura:** el texto llega por bloques. El tiempo total es el mismo; la **percepción** no: el usuario ve la primera palabra en fracciones de segundo. En una UI, cada fragmento se pinta en su línea de chat.


## 2.6-2.7 · Contexto y caché (demo conceptual con números)

Nada de API: la idea es razonar la arquitectura. Un asistente de soporte con 12k tokens de system prompt (instrucciones + catálogo) por usuario, con 20 turnos de 300 tokens:

| Estrategia | Tokens de entrada/turno |
|---|---|
| Sin caché, sin gestión | 12,000 + historial completo |
| Caché de prompt (stable 12k) | ~300 (turno) + 12k a coste reducido |
| Resumen + cache | ~300 + 1.5k (resumen) + cache(12k) |

**Ejercicio mental** (sin código): calcula el ahorro % con la tabla de precios de tu proveedor. La lección: la parte estable va al principio (cachable); lo variable (usuario), al final.


---

## Resumen del notebook

- System prompt = reglas accionables + escape; nada de "sé amable".
- Pydantic schema en el prompt + validación + re-llamada = salidas que no rompen producción.
- Tool use = el modelo pide, tú ejecutas, devuelves `tool_result` — y el modelo sigue.
- Streaming = UX; caché = coste; gestión de contexto = memoria.

---

## Ejercicios (5) — hazlo para la entrega

**1.** Rompe el asistente: pide el **estado del tiempo**. ¿El modelo responde la frase de escape exacta? Si no, ajusta el system prompt hasta que la respete en 3 pruebas de 3.

**2.** Añade `fecha_deseada: str | None` al modelo `Pedido`. Prueba con un email sin fecha: ¿el modelo la deja `null`? ¿Valida Pydantic?

**3.** Declara una segunda herramienta `listar_pedidos(cliente_id)` y prueba la conversación "¿qué pedidos tengo abiertos?". El modelo debe encadenar dos herramientas (listar → get). Súbelo con su schema.

**4.** Rompe a propósito la extracción: email con "quiero 0 tazas". ¿Qué ValidationErrror de Pydantic aparece? ¿Qué pasa en el re-llamado? Con cláusula explícita en el prompt ("si el usuario pide 0 unidades, responde amablemente que mínimo 1"), ¿funciona mejor?

**5.** *(Sin API)* Escribe el **presupuesto del asistente**: asistende con 12k de system prompt, 20 turnos por usuario, 500 usuarios/mes. Con y sin caché, ¿cuánto cambia? Usa los precios de la API que uses. Entrega la tabla.
